# 02 · SOP 传统基线：训练、嵌套验证与鲁棒性测试

**先运行01，再从上到下运行本文件。辅助文件 `sop_baselines.py` 和 `sop_features.py` 与本文件放在一起，不需要单独运行。**

本轮不使用 ResNet，也不需要下载权重：先跑 Logistic Regression（LR）与 RBF SVM。

| 特征条件 | 含义 | 维度 |
|---|---|---:|
| HSV48 | 全局颜色直方图 | 48 |
| HSV_HOG192 | 全局颜色＋4×4区域的梯度方向直方图 | 192 |

这里的梯度特征是简化的空间直方图，不是完整标准 HOG。两种条件都通过图片对的逐维绝对差值，判断两张图是不是同一商品。ResNet512／四区域2048与 SupCon 后续单独实现。

**老师反馈补充：**除了部分遮挡，同时测试噪声、模糊与旋转；报告文献综述还需解释所选近期论文与课堂内容的区别。

这两个基线不能代替作业要求的三个算法，也不能单独构成完整作业。


## 1. 导入辅助模块与定位项目

在 PyCharm 选择 `Unimelb_SML_A2` 对应的解释器／Notebook kernel。
如果 PyCharm 的 Notebook 支持不可用，可以在已激活的环境中执行 `jupyter lab`，从浏览器打开本文件。


In [ ]:
from pathlib import Path
import sys
import json
import copy
import pandas as pd
from IPython.display import display, Image as DisplayImage

cwd = Path.cwd().resolve()
module_dir = next((p for p in [cwd, cwd / 'notebooks', *cwd.parents]
                   if (p / 'sop_baselines.py').is_file()), None)
if module_dir is None:
    raise FileNotFoundError('请把 sop_baselines.py、sop_features.py 与本 Notebook 放在一起。')
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

from sop_baselines import DEFAULT_CONFIG, load_preprocessed, run_experiment

PROJECT_ROOT = module_dir.parent if module_dir.name == 'notebooks' else module_dir
print('Python:', sys.executable)
print('项目目录:', PROJECT_ROOT)


## 2. 设置输入路径与实验参数

默认使用官方训练部分中至少10,000张合格原图，按商品完整保留图片，并兼顾12个大类。
这是控制计算量的子集实验；每次训练还限制为最多1,000个锚点商品／约3,000个图片对，不能在报告中写成“用全部图片训练”。

- 外层：手写10折商品划分。内层：手写3折商品划分。
- 三个 C 候选值；边界胜出时仅根据内层验证尝试扩展范围。
- 默认额外做一次官方 test 测试；它是补充，不是10折误差条的来源。
- 一个扰动水平对应一个条件，每次只扰动查询图，另一张参考图保持干净。
- 若运行中断，可以用完全相同的代码、配置与数据重新运行，复用已完成外层折。
- **更改参数或代码时，换一个新的 OUTPUT_DIR。**

所有条件在看到结果前固定。当前是首轮传统基线方案，不能把试跑结果包装成已经完成所有正式实验。


In [ ]:
DATA_DIR = PROJECT_ROOT / 'processed_sop'
OUTPUT_DIR = PROJECT_ROOT / 'results_traditional_v1'

# 如果01的输出目录不同，只修改上面的 DATA_DIR。
CONFIG = copy.deepcopy(DEFAULT_CONFIG)

# 可选：使用全部清洗后的官方训练图片，会增加特征缓存和采样的成本。
# CONFIG['use_all_official_train'] = True

# 可选：先不做额外官方test，仍然保留完整10折外层测试。
# CONFIG['run_official_test'] = False

print(json.dumps(CONFIG, ensure_ascii=False, indent=2))
print('数据目录:', DATA_DIR)
print('结果目录:', OUTPUT_DIR)


## 3. 查看01输出是否齐全

这里读取已经构建的192维特征与清单，不重新解压 ZIP。
如果找不到文件，请先完成01的所有单元格，并检查它打印的输出路径。


In [ ]:
manifest, original_features = load_preprocessed(DATA_DIR)
print('已清洗图片数:', len(manifest))
print('商品数:', manifest['class_id'].nunique())
print('逐图特征矩阵:', original_features.shape)
display(manifest.groupby('source_split').agg(
    images=('image_id', 'size'), products=('class_id', 'nunique')))
display(manifest.head())
del original_features


## 4. 明确测试条件与评价方式

| 条件 | 具体操作 |
|---|---|
| clean | 无扰动 |
| occlusion25 | 灰色矩形遮挡约25%的实际图片内容面积 |
| noise005 | [0,1]像素尺度上加入标准差0.05的高斯噪声，再截断 |
| blur15 | 高斯模糊，Pillow radius=1.5像素 |
| rotation15 | 固定旋转+15°，保持画布尺寸，白色填充；可能裁掉边缘 |

扰动作用于白色补边内部的真实内容区域，不把生成图片计入原始数据数量。噪声／遮挡按图片编号固定随机种子。不同方法使用完全相同的商品划分、图片对与扰动。

**训练只用干净图片。** C 的选择依据是三折内层干净验证集的平均 ROC-AUC。
匹配阈值根据获选 C 的内层折外预测，目标是达到95%的验证正例召回率；之后固定阈值，在所有测试条件下使用。

**测试召回率不保证等于95%。** 应同时看实际 TPR 与 FPR，不能把结果直接称为“测试95%召回率下的FPR”。内层多个模型的分数尺度与最终重训模型可能不同，是阈值迁移的局限。

每个查询商品产生：同商品正例、同大类随机负例、按干净HSV颜色距离选择的负例。
后者是“颜色相似”的代理，不能声称它必然在语义上相似。分别报告两种负例结果，避免只选一种负例得到片面结论。

指标全部手写：混淆矩阵、Accuracy、Precision、TPR/Recall、FPR、F1、Balanced Accuracy、ROC-AUC、Average Precision。
误差条为10个外层折的**均值±1个标准差**，不是独立样本置信区间。


## 5. 开始训练与测试

**这是耗时最长的单元格。** 先生成并缓存四种扰动的传统特征，再依次完成10个外层折的四种算法／特征配置。
它会打印缓存和外层折的进度，不需要再运行单独的测试 Notebook。

官方 test 图片及其结果不参与 C 或阈值的选择。出现收敛错误时，先修复训练设置并使用新的结果目录；不要忽略错误后报告分数。

尚未在你的电脑上执行本轮实验，不能预先保证耗时、收敛、网格中间值胜出或性能。


In [ ]:
outer_results = run_experiment(DATA_DIR, OUTPUT_DIR, CONFIG)
display(outer_results.head())


## 6. 查看结果表与误差条

先比较同一算法的 HSV48 与 HSV_HOG192：

- FPR 下降：不同商品被误认成同一商品的比例减少。
- TPR 下降：同商品正确匹配能力下降，需要讨论这个代价。
- F1、ROC-AUC：辅助描述分类与分数排序的整体表现。

不能只说“数值更高所以更好”。结合扰动类型、颜色／梯度信息与误差条解释。


In [ ]:
summary = pd.read_csv(OUTPUT_DIR / 'summary_mean_sd.csv')
main = summary[summary['metric'].isin(['FPR', 'TPR', 'F1', 'ROC_AUC'])].copy()
main['mean ± SD'] = main.apply(
    lambda row: f"{row['mean']:.4f} ± {row['sd_across_outer_folds']:.4f}", axis=1)
display(main.pivot(index=['algorithm', 'features', 'negative_subset', 'condition'],
                   columns='metric', values='mean ± SD'))

for filename in ['random_robustness.png', 'colour_hard_robustness.png']:
    display(DisplayImage(filename=str(OUTPUT_DIR / 'figures' / filename)))


## 7. 检查超参数范围与局部信息的作用

老师要求：每个超参数至少三个候选值，且多数时候应选到中间值。
代码根据内层验证尝试扩展范围，仍未解决时会明确记录，不能为满足要求强行选择中间值。
若需要扩展，依据内层验证日志调整，在新的结果目录运行；不要依据外层／官方test分数挑范围。

`delta_local_minus_global`：192维结果减去48维结果。
因此 FPR 的负差值通常有利；TPR 的负差值表示匹配能力有损失。
这也是同折配对比较，不能把几千个相关图片对当作几千次独立实验。


In [ ]:
audit = json.loads((OUTPUT_DIR / 'grid_requirement_audit.json').read_text(encoding='utf-8'))
display(pd.DataFrame(audit))
display(pd.read_csv(OUTPUT_DIR / 'paired_delta_mean_sd.csv'))

official_path = OUTPUT_DIR / 'official_holdout' / 'metrics.csv'
if official_path.exists():
    print('下面是额外官方holdout结果：只有一次测试，不用于生成CV误差条。')
    display(pd.read_csv(official_path))


## 8. 写报告与下一步

报告可以说明：原图资格检查、缺失／不可读样本与单样本商品处理、保持比例缩放与补边、HSV颜色与空间梯度特征、图片对构建、训练折内标准化、商品分组嵌套交叉验证、C调参、验证阈值、四种扰动、手写指标和跨折误差条。

必须报告实际剔除数量、选中图片／商品数量、图片对采样上限、扰动参数、负例选择机制、实际C选择及未解决网格问题，不能使用预估值代替运行结果。

老师反馈还要求：在文献综述中解释所选论文如何不同于课堂内容。对 SupCon，应重点说明**监督对比学习的目标、同标签正例构建以及学习匹配表示的方式**，与课件已有的 NN/CNN/ResNet 架构、预训练和微调作区分，并引用论文。截图未显示论文标题，不能仅凭截图认定具体论文已获批准。

下一步再加入固定预训练 ResNet 全局／区域特征和 SupCon。它们必须复用本轮保存的商品折、图片对编号与扰动规则才能公平比较。固定特征可以缓存，SOP上学习的表示必须在各训练折内重新训练。

**本轮不会修改01，也不会自动上传到GitHub。数据、缓存和模型留在本机。** 提交时仍需三算法、至少两篇相关文献、四页报告、团队会议与代码运行说明；传统基线完成不代表整个作业完成。
